# DevGen TrOCR LoRA Fine-tuning (8 Epochs)

Trains a fresh LoRA adapter on `paudelanil/trocr-devanagari-2` (Devanagari base).

**Setup:** Settings → GPU T4 x2 → Internet ON → Run All

In [ ]:
%pip install -q transformers peft accelerate datasets editdistance

In [ ]:
import os, io, random, shutil
import torch
import editdistance
from PIL import Image
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    TrOCRProcessor,
    ViTImageProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
)
from peft import get_peft_model, LoraConfig

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
MODEL_NAME = "paudelanil/trocr-devanagari-2"
DATASET_NAME = "c3rl/IIIT-INDIC-HW-WORDS-Hindi"
OUTPUT_DIR = "./trocr-devanagari-lora"

BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 8
LEARNING_RATE = 5e-5
EVAL_STEPS = 1000
SAVE_STEPS = 1000
LOGGING_STEPS = 100
MAX_TARGET_LENGTH = 128
EVAL_LIMIT = 2000

In [ ]:
# ═══════════════════════════════════════════════════════════
#  LOAD PROCESSOR (handles the paudelanil compatibility fix)
# ═══════════════════════════════════════════════════════════
#
# WHY we load image_processor and tokenizer separately:
#
# paudelanil's model was uploaded with an older transformers version.
# Its preprocessor_config.json is missing the 'image_processor_type' key
# that newer transformers requires.
#
# The fix: load the two pieces separately and combine them.
#   - Image processor: from google/vit-base-patch16-224-in21k (224x224 size)
#   - Tokenizer: from paudelanil/trocr-devanagari-2 (Nepali NepBERT vocab)
#
# DO NOT use microsoft/trocr-base-handwritten for the tokenizer!
# That has an English vocabulary. paudelanil's decoder expects Nepali tokens.
# ═══════════════════════════════════════════════════════════

print("Loading image processor (ViT 224x224)...")
image_processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224-in21k"
)

print("Loading tokenizer (Nepali NepBERT from paudelanil)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

processor = TrOCRProcessor(
    image_processor=image_processor,
    tokenizer=tokenizer
)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print("Processor ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  LOAD MODEL + APPLY LoRA
# ═══════════════════════════════════════════════════════════
print("Loading model weights...")
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = tokenizer.sep_token_id
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.max_length = MAX_TARGET_LENGTH
model.generation_config.num_beams = 4

print("Applying LoRA...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value", "key", "dense"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ═══════════════════════════════════════════════════════════
#  LOAD DATASET
# ═══════════════════════════════════════════════════════════
print("Loading dataset...")
ds = load_dataset(DATASET_NAME)
print(f"Train: {len(ds['train'])} | Val: {len(ds['validation'])} | Test: {len(ds['test'])}")

def preprocess_batch(examples):
    images = []
    for img_data in examples['image']:
        if isinstance(img_data, bytes):
            images.append(Image.open(io.BytesIO(img_data)).convert('RGB'))
        elif isinstance(img_data, dict) and 'bytes' in img_data:
            images.append(Image.open(io.BytesIO(img_data['bytes'])).convert('RGB'))
        else:
            images.append(img_data.convert('RGB'))
    pixel_values = processor(images, return_tensors='pt').pixel_values
    labels = tokenizer(
        examples['text'], padding='max_length',
        max_length=MAX_TARGET_LENGTH, truncation=True
    ).input_ids
    labels = [
        [l if l != tokenizer.pad_token_id else -100 for l in row]
        for row in labels
    ]
    return {'pixel_values': pixel_values, 'labels': labels}

train_dataset = ds['train'].with_transform(preprocess_batch)
eval_ds = ds['validation'].select(range(min(EVAL_LIMIT, len(ds['validation']))))
eval_dataset = eval_ds.with_transform(preprocess_batch)
print(f"Using {len(train_dataset)} train, {len(eval_dataset)} eval samples")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  TRAIN
# ═══════════════════════════════════════════════════════════
def compute_metrics(pred):
    labels_ids = pred.label_ids.copy()
    pred_ids = pred.predictions
    labels_ids[labels_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_str = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)
    cer_sum, valid = 0, 0
    for p, l in zip(pred_str, labels_str):
        if len(l) > 0:
            cer_sum += editdistance.eval(p, l) / len(l)
            valid += 1
    return {'cer': cer_sum / valid if valid > 0 else 0.0}

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    predict_with_generate=True,
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    remove_unused_columns=False,
    fp16=True,
    dataloader_pin_memory=True,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    processing_class=image_processor,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

spe = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Training: {EPOCHS} epochs, ~{spe} steps/epoch, ~{spe * EPOCHS} total")
trainer.train()

In [ ]:
# ═══════════════════════════════════════════════════════════
#  SAVE + EVALUATE
# ═══════════════════════════════════════════════════════════
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

metrics = trainer.evaluate()
print(f"Final CER: {metrics['eval_cer']:.4f}  (brother's was 0.1397)")
if metrics['eval_cer'] < 0.14:
    print("YOU BEAT YOUR BROTHER'S MODEL!")
else:
    print("Close — try more epochs or lower LR next time")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  QUICK TEST
# ═══════════════════════════════════════════════════════════
test_ds = load_dataset(DATASET_NAME, split='test')
random.seed(42)
indices = random.sample(range(len(test_ds)), 10)

model.eval()
total_cer = 0
print('Test predictions (10 samples):')
for idx in indices:
    sample = test_ds[idx]
    img = sample['image'] if isinstance(sample['image'], Image.Image) else Image.open(io.BytesIO(sample['image']['bytes']))
    pv = processor(images=img.convert('RGB'), return_tensors='pt').pixel_values.to(model.device)
    with torch.no_grad():
        gen = model.generate(pv, max_length=128, num_beams=4)
    pred = tokenizer.decode(gen[0], skip_special_tokens=True)
    gt = sample['text']
    cer = editdistance.eval(pred, gt) / len(gt) if gt else 0
    total_cer += cer
    print(f"  GT: {gt:20s} | Pred: {pred:20s} | CER: {cer:.2f}")
print(f"Avg CER: {total_cer/10:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  ZIP FOR DOWNLOAD
# ═══════════════════════════════════════════════════════════
zip_path = shutil.make_archive('trocr-devanagari-lora', 'zip', '.', 'trocr-devanagari-lora')
print(f'Model zipped: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('Download from Output tab -> unzip into DevGen/trocr-devanagari-lora/')